# NMF + GMM Tissue Classification Pipeline

Processes raw Raman hyperspectral biopsy cubes `(rows, cols, λ)` to produce per-pixel binary tumor masks.

**Pipeline:**
1. **Spatial 3×3 median filter** — removes noise and cosmic-ray spikes per wavelength channel
2. **Normalization** — shifts and clips each spectrum to [0, 1]
3. **NMF decomposition** — learns `NMF_COMPONENTS` spectral basis functions; the abundance matrix W feeds the GMM
4. **GMM clustering** — partitions strong-signal pixels into `GMM_N_CLUSTERS` tissue classes
5. **Manual cluster assignment** — cluster IDs are recorded in `data/processed/file_names.csv`; the batch section reads these to generate binary tumor masks

**Inputs:** `data/raw/LGm-*/*.npy`, `data/RamanFreqTable.csv`, `data/processed/file_names.csv`  
**Outputs:** `{OUTPUT_DIR}/{patient_id}/tumor_mask.npy`, `results/exp_102/tumor_maps.npy`

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import median_filter
from sklearn.decomposition import NMF
from sklearn.mixture import GaussianMixture
from matplotlib.colors import ListedColormap

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

# ── Paths ──────────────────────────────────────────────
RAW_DATA_DIR  = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')   # source: contains file_names.csv
OUTPUT_DIR    = Path('../data/processed_test')   # change to a new dir to avoid overwriting prior results
RESULTS_DIR   = Path('../results/exp_102')
CALIB_PATH    = Path('../data/RamanFreqTable.csv')
PLOTS_DIR     = RESULTS_DIR / 'cluster_plots'

# ── Hyperparameters ────────────────────────────────────
NMF_COMPONENTS     = 100   # number of NMF basis spectra
GMM_N_CLUSTERS     = 3     # tissue classes (junk / healthy / tumor)
MEDIAN_KERNEL      = 3     # spatial median filter footprint (pixels)
WEAK_SIGNAL_THRESH = 0.6   # pixels below this max-intensity are masked before GMM
RANDOM_STATE       = 42

# ── Cluster colours (Blue, Green, Red) ─────────────────────
CLUSTER_COLORS = ['#1f77b4', '#2ca02c', '#d62728']

# ── Wavenumber calibration ─────────────────────────────────
wavenumbers = pd.read_csv(CALIB_PATH)['freq'].values
print(f"Wavenumber range: [{wavenumbers.min():.1f}, {wavenumbers.max():.1f}] cm\u207b\u00b9  ({len(wavenumbers)} channels)")

---
## Section 1: Single-Slide Walkthrough

The following cells run the full pipeline on one representative slide.
All parameters are defined in the configuration cell above.

In [ ]:
slide_path = sorted(RAW_DATA_DIR.rglob('*.npy'))[1]
data_cube  = np.load(slide_path)
X, Y, Lambda = data_cube.shape
slide_id   = slide_path.stem

print(f"Slide:  {slide_path.relative_to(RAW_DATA_DIR.parent)}")
print(f"Shape:  {data_cube.shape}  (rows \u00d7 cols \u00d7 \u03bb)")
print(f"Range:  [{data_cube.min():.1f}, {data_cube.max():.1f}]")

cx, cy = X // 2, Y // 2
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(wavenumbers, data_cube[cx, cy, :], lw=1)
ax.set_xlabel('Raman shift (cm\u207b\u00b9)')
ax.set_ylabel('Intensity (counts)')
ax.set_title(f'Raw spectrum \u2014 centre pixel ({cx}, {cy}) of {slide_id}')
ax.grid(True, alpha=0.3)
plt.tight_layout()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
fig_path = RESULTS_DIR / f'raw_spectrum_{slide_id}.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Apply 3x3 spatial median filter per wavelength ─────────────────────
data_filtered = np.zeros_like(data_cube)
for i in range(Lambda):
    data_filtered[:, :, i] = median_filter(data_cube[:, :, i], size=MEDIAN_KERNEL, mode='reflect')

print(f"Median filtering complete  ({Lambda} wavelength channels)")

# ── Biological RGB helper (R=Phe, G=C-H stretch, B=CH2) ─────────────────
def _bio_rgb(cube, wn):
    def band_sum(lo, hi):
        return cube[:, :, np.argmin(np.abs(wn - lo)):np.argmin(np.abs(wn - hi))].sum(axis=2)
    def norm(c):
        c = c - c.min()
        return c / (c.max() + 1e-8)
    R = norm(band_sum(990,  1020))   # Phenylalanine
    G = norm(band_sum(2800, 3000))   # C-H stretch (lipids)
    B = norm(band_sum(1430, 1460))   # CH2 deformation
    return np.power(np.stack([R, G, B], axis=-1), 0.8)

slide_rgb = _bio_rgb(data_filtered, wavenumbers)

# ── Publication figure: overview | crop+box | spectrum ───────────────────
CROP_PAD = 8   # pixels on each side of centre pixel in the crop panel
r0, r1 = max(cx - CROP_PAD, 0), min(cx + CROP_PAD + 1, X)
c0, c1 = max(cy - CROP_PAD, 0), min(cy + CROP_PAD + 1, Y)

fig, (ax_ov, ax_crop, ax_spec) = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: full slide in biological RGB + rectangle showing crop region
ax_ov.imshow(slide_rgb, aspect='equal')
crop_rect = mpatches.Rectangle(
    (c0, r0), c1 - c0, r1 - r0,
    linewidth=1.5, edgecolor='yellow', facecolor='none'
)
ax_ov.add_patch(crop_rect)
ax_ov.set_title('(a) Slide (R=Phe, G=Lipid, B=CH₂)\nYellow box = zoomed region', fontsize=14)
ax_ov.set_xlabel('Column (pixels)', fontsize=12)
ax_ov.set_ylabel('Row (pixels)', fontsize=12)
ax_ov.tick_params(axis='both', labelsize=10)

# Panel 2: cropped zoom with 3x3 kernel box
crop_rgb = slide_rgb[r0:r1, c0:c1]
ax_crop.imshow(crop_rgb, aspect='equal')
box_rect = mpatches.Rectangle(
    (cy - c0 - 1.5, cx - r0 - 1.5), MEDIAN_KERNEL, MEDIAN_KERNEL,
    linewidth=2, edgecolor='cyan', facecolor='none', label='3×3 kernel'
)
ax_crop.add_patch(box_rect)
ax_crop.plot(cy - c0, cx - r0, 'c+', ms=6)   # centre pixel marker
ax_crop.legend(loc='upper right', fontsize=11)
ax_crop.set_title(f'(b) Zoomed crop ({2*CROP_PAD+1}×{2*CROP_PAD+1} px)\nCyan box = 3×3 median filter kernel', fontsize=14)
ax_crop.set_xlabel('Column', fontsize=12)
ax_crop.set_ylabel('Row', fontsize=12)
ax_crop.tick_params(axis='both', labelsize=10)

# Panel 3: raw vs filtered spectrum at centre pixel
ax_spec.plot(wavenumbers, data_cube[cx, cy, :], lw=1, alpha=0.7, label='Raw')
ax_spec.plot(wavenumbers, data_filtered[cx, cy, :], lw=1.5, label='Filtered')
ax_spec.set_xlabel('Raman shift (cm⁻¹)', fontsize=12)
ax_spec.set_ylabel('Intensity (counts)', fontsize=12)
ax_spec.set_title(f'(c) Spectrum at centre pixel ({cx}, {cy})', fontsize=14)
ax_spec.legend(fontsize=11)
ax_spec.tick_params(axis='both', labelsize=10)
ax_spec.grid(True, alpha=0.3)

plt.tight_layout()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
fig_path = RESULTS_DIR / 'median_filter_illustration.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

In [ ]:
data_flat = data_filtered.reshape(-1, Lambda)

# Shift by global minimum to ensure non-negativity, then scale to [0, 1]
data_norm = data_flat - data_flat.min() + 1e-8
data_norm = np.clip(data_norm / np.percentile(data_norm, 99.9), 0, 1)

print(f"Flattened:  {data_flat.shape}")
print(f"Normalized: [{data_norm.min():.4f}, {data_norm.max():.4f}]")

In [ ]:
print(f"Fitting NMF ({NMF_COMPONENTS} components, 500 iterations, init=nndsvda)...")
nmf = NMF(n_components=NMF_COMPONENTS, init='nndsvda', max_iter=500, random_state=RANDOM_STATE)
W_nmf = nmf.fit_transform(data_norm)
H_nmf = nmf.components_

print(f"W (abundances): {W_nmf.shape}")
print(f"H (bases):      {H_nmf.shape}")
print(f"Reconstruction error: {nmf.reconstruction_err_:.4e}")

### 1.4 GMM Tissue Classification

GMM is applied only to **strong-signal pixels** (normalised max intensity ≥ `WEAK_SIGNAL_THRESH`),
so that low-SNR pixels do not distort the tissue-class centroids.

After running the next cell, **visually inspect the saved plots** and record your cluster assignments
in `data/processed/file_names.csv` (column `is_tumor`: comma-separated cluster indices that
correspond to good-quality / tumour signal, e.g. `"0,2"`).  
Two clusters may be assigned when two GMM components are spatially interleaved with similar
spectral character — both are treated as tumour-positive pixels.

In [ ]:
cmap = ListedColormap(CLUSTER_COLORS)

# ── Fit GMM on strong-signal pixels only ────────────────────────────────
mask_strong = data_norm.max(axis=1) >= WEAK_SIGNAL_THRESH
W_strong    = W_nmf[mask_strong]

print(f"Strong-signal pixels: {mask_strong.sum():,} / {len(mask_strong):,}  ({mask_strong.mean()*100:.1f}%)")

gmm = GaussianMixture(n_components=GMM_N_CLUSTERS, n_init=10, random_state=RANDOM_STATE)
gmm.fit(W_strong)

soft_labels = gmm.predict_proba(W_strong)
hard_strong = np.argmax(soft_labels, axis=1)

# Recompose into full spatial grid (weak pixels = -1)
hard_full = np.full(X * Y, -1, dtype=np.int8)
hard_full[mask_strong] = hard_strong
hard_labels_2d = hard_full.reshape(X, Y)

print("Cluster summary:")
for k in range(GMM_N_CLUSTERS):
    n = (hard_strong == k).sum()
    print(f"  Cluster {k} ({CLUSTER_COLORS[k]}): {n:,} pixels  (GMM weight {gmm.weights_[k]:.1%})")

# ── Publication figure: cluster map + mean spectrum per cluster ───────────
slide_plot_dir = PLOTS_DIR / slide_id
slide_plot_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Panel 1: spatial cluster map
ax_map = axes[0]
masked_labels = np.ma.masked_where(hard_labels_2d == -1, hard_labels_2d)
im = ax_map.imshow(masked_labels, cmap=cmap, vmin=0, vmax=GMM_N_CLUSTERS - 1, aspect='equal')
ax_map.imshow((hard_labels_2d == -1).astype(float), cmap='Greys', alpha=0.4, aspect='equal')
cbar = plt.colorbar(im, ax=ax_map, ticks=range(GMM_N_CLUSTERS))
cbar.set_ticklabels([f'Cluster {k}' for k in range(GMM_N_CLUSTERS)])
ax_map.set_title(f'GMM Cluster Map {slide_id}')
ax_map.set_xlabel('Column'); ax_map.set_ylabel('Row')

# Panels 2–4: mean ± std spectrum per cluster
for k in range(GMM_N_CLUSTERS):
    ax = axes[k + 1]
    mask_k    = hard_strong == k
    spectra_k = data_norm[mask_strong][mask_k]
    mean_spec = spectra_k.mean(axis=0)
    std_spec  = spectra_k.std(axis=0)
    ax.plot(wavenumbers, mean_spec, lw=1.5, color=CLUSTER_COLORS[k])
    ax.fill_between(wavenumbers, mean_spec - std_spec, mean_spec + std_spec, alpha=0.25, color=CLUSTER_COLORS[k])
    ax.set_title(f'Cluster {k}  ({mask_k.sum():,} px)')
    ax.set_xlabel('Raman shift (cm⁻¹)')
    if k == 0:
        ax.set_ylabel('Intensity (norm.)')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'GMM clustering — {slide_id}')
plt.tight_layout()
plt.savefig(slide_plot_dir / 'cluster_map_and_spectra.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved to: {slide_plot_dir / 'cluster_map_and_spectra.png'}")
print("Record tumour cluster IDs in data/processed/file_names.csv (column: is_tumor).")

In [ ]:
# Requires file_names.csv to have an entry for this slide (column: is_tumor)
assignments = pd.read_csv(PROCESSED_DIR / 'file_names.csv').set_index('patient')

if slide_id not in assignments.index:
    print(f"No entry for '{slide_id}' in file_names.csv yet.")
    print("Add a row with is_tumor = comma-separated cluster IDs and re-run this cell.")
else:
    tumor_clusters = [int(c) for c in str(assignments.loc[slide_id, 'is_tumor']).split(',') if c.strip()]
    print(f"Tumour clusters for {slide_id}: {tumor_clusters}")

    tumor_mask_demo = np.isin(hard_labels_2d, tumor_clusters)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(np.ma.masked_where(hard_labels_2d == -1, hard_labels_2d),
                   cmap=cmap, vmin=0, vmax=GMM_N_CLUSTERS - 1, aspect='equal')
    axes[0].set_title('GMM clusters')
    axes[1].imshow(tumor_mask_demo, cmap='Reds', aspect='equal')
    axes[1].set_title(f'Tumour mask  (clusters {tumor_clusters})')
    for ax in axes:
        ax.set_xlabel('Column'); ax.set_ylabel('Row')
    plt.tight_layout()
    plt.show()

---
## Section 2: Diagnostic Validation

The plots below justify the parameter choices made in Section 1.
They operate on the demo slide processed above.

In [ ]:
max_vals = data_norm.max(axis=1)

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(max_vals, bins=60, color='steelblue', alpha=0.75, edgecolor='none')
ax.axvline(WEAK_SIGNAL_THRESH,     color='red',    lw=2,   ls='--', label=f'WEAK_SIGNAL_THRESH = {WEAK_SIGNAL_THRESH}')
ax.axvline(max_vals.mean(),        color='orange', lw=1.5, ls=':',  label=f'Mean = {max_vals.mean():.3f}')
ax.axvline(np.median(max_vals),    color='green',  lw=1.5, ls=':',  label=f'Median = {np.median(max_vals):.3f}')
ax.set_xlabel('Max normalised intensity per pixel')
ax.set_ylabel('Pixel count')
ax.set_title('(e) Distribution of per-pixel signal strength')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
fig_path = RESULTS_DIR / 'max_vals_distribution_img_0.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

below = (max_vals < WEAK_SIGNAL_THRESH).sum()
print(f"Pixels below threshold: {below:,} / {len(max_vals):,}  ({below/len(max_vals)*100:.1f}%) \u2014 excluded from GMM")

In [ ]:
mask_weak_2d = (~mask_strong).reshape(X, Y)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(mask_weak_2d, cmap='RdYlGn_r', aspect='equal')
ax.set_title(
    f'Weak-signal mask (max < {WEAK_SIGNAL_THRESH})\n'
    f'{mask_weak_2d.sum():,} pixels masked  ({mask_weak_2d.mean()*100:.1f}%)'
)
ax.set_xlabel('Column'); ax.set_ylabel('Row')
plt.tight_layout()
plt.show()

In [ ]:
# Requires file_names.csv entry for the demo slide (see Cell 9)
if slide_id not in assignments.index:
    print("Update file_names.csv first (see Cell 9).")
else:
    tumor_clusters_demo = [int(c) for c in str(assignments.loc[slide_id, 'is_tumor']).split(',') if c.strip()]

    fig, (ax_map, ax_spec) = plt.subplots(
        1, 2, figsize=(16, 4),
        gridspec_kw={'width_ratios': [1, 3]}
    )

    # Left (0.25): spatial cluster map
    masked_labels = np.ma.masked_where(hard_labels_2d == -1, hard_labels_2d)
    im = ax_map.imshow(masked_labels, cmap=cmap, vmin=0, vmax=GMM_N_CLUSTERS - 1, aspect='equal')
    ax_map.imshow((hard_labels_2d == -1).astype(float), cmap='Greys', alpha=0.4, aspect='equal')
    cbar = plt.colorbar(im, ax=ax_map, ticks=range(GMM_N_CLUSTERS), shrink=0.8)
    cbar.set_ticklabels([f'Cluster {k}' for k in range(GMM_N_CLUSTERS)], fontsize=10)
    cbar.ax.tick_params(labelsize=10)
    ax_map.set_title(f'(c) Tissue clusters', fontsize=14)
    ax_map.set_xlabel('Column', fontsize=12)
    ax_map.set_ylabel('Row', fontsize=12)
    ax_map.tick_params(axis='both', labelsize=10)

    # Right (0.75): overlaid mean ± std spectra, tumour clusters drawn bolder
    for k in range(GMM_N_CLUSTERS):
        mask_k    = hard_strong == k
        spectra_k = data_norm[mask_strong][mask_k]
        mean_spec = spectra_k.mean(axis=0)
        std_spec  = spectra_k.std(axis=0)
        label = f'Cluster {k} (tumour)' if k in tumor_clusters_demo else f'Cluster {k}'
        lw    = 2.5 if k in tumor_clusters_demo else 1.5
        ax_spec.plot(wavenumbers, mean_spec, lw=lw, color=CLUSTER_COLORS[k], label=label)
        ax_spec.fill_between(wavenumbers, mean_spec - std_spec, mean_spec + std_spec,
                             alpha=0.15, color=CLUSTER_COLORS[k])
    ax_spec.set_xlabel('Raman shift (cm⁻¹)', fontsize=12)
    ax_spec.set_ylabel('Intensity (norm.)', fontsize=12)
    ax_spec.set_title(f'(d) Mean spectra per cluster', fontsize=14)
    ax_spec.legend(fontsize=11)
    ax_spec.tick_params(axis='both', labelsize=10)
    ax_spec.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_path = RESULTS_DIR / 'publication_cluster_spectra.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'Saved: {fig_path}')
    plt.show()

---
## Section 3: Batch Processing (All Slides)

`process_slide()` applies the same pipeline (filter → normalise → NMF → GMM) to a single `.npy` file
and returns hard cluster labels as a `(rows, cols)` array.

**To avoid overwriting previous results**, set `OUTPUT_DIR` in the configuration cell to a new path.

In [ ]:
def process_slide(slide_path, wavenumbers, output_dir=None):
    """Filter, normalise, NMF, GMM -> hard cluster labels (rows, cols).

    If output_dir is given, also persists the artifacts the modelling pipeline
    reads directly (src/data/preprocessing.py): hard_labels.npy,
    strong_signal_mask.npy and metadata.json under output_dir/<slide_name>/.
    Previously only an archived script (_archive/dev_scripts/
    batch_process_all_slides.py) wrote these, even though data/processed/<slide>/
    already holds them from that script's earlier run -- this notebook's own
    batch loop did not, so it could not regenerate its own claimed outputs from
    raw data alone. This closes that gap.
    """
    data = np.load(slide_path)
    X_s, Y_s, L = data.shape

    filtered = np.zeros_like(data)
    for i in range(L):
        filtered[:, :, i] = median_filter(data[:, :, i], size=MEDIAN_KERNEL, mode='reflect')

    flat = filtered.reshape(-1, L)
    norm = flat - flat.min() + 1e-8
    norm = np.clip(norm / np.percentile(norm, 99.9), 0, 1)

    nmf_s = NMF(n_components=NMF_COMPONENTS, init='nndsvda', max_iter=500, random_state=RANDOM_STATE)
    W = nmf_s.fit_transform(norm)

    mask = norm.max(axis=1) >= WEAK_SIGNAL_THRESH
    gmm_s = GaussianMixture(n_components=GMM_N_CLUSTERS, n_init=10, random_state=RANDOM_STATE)
    gmm_s.fit(W[mask])

    hard = np.full(X_s * Y_s, -1, dtype=np.int8)
    hard[mask] = gmm_s.predict(W[mask])
    hard_2d = hard.reshape(X_s, Y_s)

    if output_dir is not None:
        import json
        slide_dir = Path(output_dir) / slide_path.stem
        slide_dir.mkdir(parents=True, exist_ok=True)
        np.save(slide_dir / 'hard_labels.npy', hard_2d)
        np.save(slide_dir / 'strong_signal_mask.npy', mask.reshape(X_s, Y_s))
        metadata = {
            'slide_name': slide_path.stem,
            'raw_file': str(slide_path),
            'shape': {'X': int(X_s), 'Y': int(Y_s), 'Lambda': int(L)},
            'total_pixels': int(X_s * Y_s),
            'strong_signal_pixels': int(mask.sum()),
            'weak_signal_pixels': int((~mask).sum()),
            'signal_threshold': WEAK_SIGNAL_THRESH,
            'nmf_components': NMF_COMPONENTS,
            'nmf_iterations': int(nmf_s.n_iter_),
            'nmf_reconstruction_error': float(nmf_s.reconstruction_err_),
            'gmm_components': GMM_N_CLUSTERS,
            'gmm_bic': float(gmm_s.bic(W[mask])),
            'gmm_weights': [float(w) for w in gmm_s.weights_],
            # No 'tissue_identification' block here: which cluster is
            # healthy/junk/tumor is a manual judgment call recorded in
            # file_names.csv's is_tumor column (Section 3 below), not something
            # to infer automatically from cluster order or mean intensity -- an
            # earlier version of this metadata did that and it disagreed with
            # the manual selection on the first slide checked during the
            # Stage 1 investigation (docs/stage1_data_layer.md).
        }
        with open(slide_dir / 'metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)

    return hard_2d

In [ ]:
# generate cluster maps for all slides, keep them in memory for labelling, and
# persist the per-slide artifacts the modelling pipeline reads directly
# (src/data/preprocessing.py): hard_labels.npy, strong_signal_mask.npy,
# metadata.json under OUTPUT_DIR/<slide_name>/.

from tqdm.auto import tqdm


def parse_clusters(val):
    return [int(c) for c in str(val).split(',') if c.strip()]


RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cluster_maps = {}   # {sample_name: hard_2d cluster map (rows, cols), -1 = weak signal}

slide_paths = sorted(RAW_DATA_DIR.rglob('*.npy'))
print(f"Processing {len(slide_paths)} slides...")

for slide_path in tqdm(slide_paths, desc="Clustering slides"):
    sample_name = slide_path.stem
    cluster_maps[sample_name] = process_slide(slide_path, wavenumbers, output_dir=OUTPUT_DIR)

print(f"Generated cluster maps for {len(cluster_maps)} slides.")
print(f"Saved hard_labels.npy / strong_signal_mask.npy / metadata.json per slide to {OUTPUT_DIR}")

All slides clustering processed. Inspect the clusters and assign the tumor clusters in  `data/processed/file_names.csv`
(`is_tumor` column: comma-separated IDs, e.g. `"0,2"` if both clusters 0 and 2 are tumour-positive).

The next cell creates the `tumor_maps.npy` that will be used later to create the hdf5 dataset.

Slides without an entry in `file_names.csv` are skipped with a warning.

In [ ]:
# Check file_names.csv for tumour cluster assignments and build per-slide tumour masks

assignments_all = pd.read_csv(PROCESSED_DIR / 'file_names.csv').set_index('patient')
tumor_maps = {}

for slide_path in slide_paths:
    sample_name = slide_path.stem
    if sample_name not in assignments_all.index:
        print(f"  [skip] {sample_name} — no entry in file_names.csv")
        continue

    hard_2d = cluster_maps[sample_name]
    tumor_clusters = parse_clusters(assignments_all.loc[sample_name, 'is_tumor'])
    tumor_mask = np.isin(hard_2d, tumor_clusters)

    out_dir = OUTPUT_DIR / sample_name
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'tumor_mask.npy', tumor_mask)

    tumor_maps[sample_name] = tumor_mask
    print(f"  {sample_name}: {tumor_mask.mean()*100:.1f}% tumour  (clusters {tumor_clusters})")

np.save(RESULTS_DIR / 'tumor_maps.npy', tumor_maps)
print(f"\nSaved {len(tumor_maps)} tumor masks to {RESULTS_DIR / 'tumor_maps.npy'}")

In [ ]:
n = len(tumor_maps)
ncols = 6
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5))
axes = np.array(axes).flatten()

for ax, (patient_id, mask) in zip(axes, sorted(tumor_maps.items())):
    ax.imshow(mask, cmap='Reds', aspect='auto')
    ax.set_title(patient_id.split('_')[0], fontsize=7)
    ax.axis('off')

for ax in axes[n:]:
    ax.axis('off')

plt.suptitle(f'Tumour masks \u2014 {n} slides', y=1.01)
plt.tight_layout()
out_png = RESULTS_DIR / 'tumor_masks_summary.png'
plt.savefig(out_png, dpi=100, bbox_inches='tight')
plt.show()
print(f'Summary grid saved to {out_png}')

---
## Section 4: Export to HDF5 Dataset

**Superseded.** The HDF5 export now lives in `src/data/preprocessing.py` (the
single producer, folded in from this section during the MIL reframing's Stage 1
-- see `docs/stage1_data_layer.md`), not in this notebook. Three things changed
from what this section used to do:

- Baseline correction is **IARPLS**, not airPLS(λ=100) -- see
  `docs/stage0_findings.md` for why.
- The exported mask is **every strong-signal pixel**, not only the
  manually-selected "tumor" clusters -- the old, narrower selection survives
  exactly as the boolean `viable_legacy` column, so anything that depended on
  it still can.
- It reads the `hard_labels.npy` / `strong_signal_mask.npy` that Section 3
  above now saves, rather than recomputing NMF/GMM (which is why the export
  itself takes minutes, not hours).

Run it from the command line, not from a notebook cell:

```bash
python scripts/build_glioma_mil.py --workers 12
```

That writes `data/processed/glioma_mil.h5` from `PROCESSED_DIR` (i.e. from
whatever Section 3 last saved to `OUTPUT_DIR` — remember `OUTPUT_DIR` and
`PROCESSED_DIR` are deliberately different paths, per the configuration cell's
comment, so a fresh Section 3 run does not silently overwrite the validated
artifacts in `data/processed/` until you decide to promote it there).

The cell below calls the same producer directly, for anyone who wants to step
through it inside this notebook instead of the command line.

In [ ]:
# Calls the Stage 1 producer directly -- see src/data/preprocessing.py.
# Equivalent to running scripts/build_glioma_mil.py from the command line.
import sys
sys.path.insert(0, str(Path('..').resolve()))

from src.data.preprocessing import RamanPreprocessor

preprocessor = RamanPreprocessor(
    raw_data_dir=RAW_DATA_DIR,
    processed_dir=PROCESSED_DIR,
    output_file=PROCESSED_DIR / 'glioma_mil.h5',
    calibration_path=CALIB_PATH,
)
preprocessor.run()

---
## Section 5: Dataset Statistics

Reports three sets of metrics computed from the exported HDF5 and the raw slide files:

1. **Per-slide selection** — number and percentage of spectra retained per slide.
2. **Class distribution** — number of slides per class (LGm-1…6) and total spectra per class.
3. **Per-fold statistics** — same breakdown for train/test splits defined in `data/splits/manual_splits.json`.

In [ ]:
import json
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

_H5_PATH     = PROCESSED_DIR / 'glioma_mil.h5'
_SPLITS_PATH = Path('../data/splits/manual_splits.json')
_RAW_DIR     = Path('../data/raw')

CLASS_NAMES = {i: f'LGm-{i+1}' for i in range(6)}

# ── Load HDF5 metadata ────────────────────────────────────────────────────
with h5py.File(_H5_PATH, 'r') as _f:
    _sample_ids = _f['sample_ids'][:].astype(str)
    _y_class    = _f['y_class'][:]

# Per-slide: spectra count and class
_slides_in_h5 = np.unique(_sample_ids)
_spectra_count = {s: int((_sample_ids == s).sum()) for s in _slides_in_h5}
_slide_class   = {s: int(_y_class[_sample_ids == s][0]) for s in _slides_in_h5}

# ── Total pixels per slide from raw .npy shapes (memory-mapped, fast) ─────
_raw_total = {}
for _p in sorted(_RAW_DIR.rglob('*.npy')):
    _arr = np.load(_p, mmap_mode='r')
    _raw_total[_p.stem] = _arr.shape[0] * _arr.shape[1]

# ══════════════════════════════════════════════════════════════════════════
# 1. Per-slide selection statistics
# ══════════════════════════════════════════════════════════════════════════
print("=" * 75)
print("1. SPECTRA SELECTION PER SLIDE")
print("=" * 75)
print(f"{'Slide':<38} {'Class':<8} {'Selected':>10} {'Total px':>10} {'%':>7}")
print("-" * 75)
for _s in sorted(_slides_in_h5):
    _cls   = CLASS_NAMES.get(_slide_class[_s], f'cls-{_slide_class[_s]}')
    _sel   = _spectra_count[_s]
    _total = _raw_total.get(_s, 0)
    _pct   = 100 * _sel / _total if _total > 0 else float('nan')
    print(f"{_s:<38} {_cls:<8} {_sel:>10,} {_total:>10,} {_pct:>6.1f}%")

_total_sel   = sum(_spectra_count.values())
_total_raw   = sum(_raw_total.get(s, 0) for s in _slides_in_h5)
_overall_pct = 100 * _total_sel / _total_raw if _total_raw > 0 else float('nan')
print("-" * 75)
print(f"{'TOTAL':<38} {'':8} {_total_sel:>10,} {_total_raw:>10,} {_overall_pct:>6.1f}%")

# ══════════════════════════════════════════════════════════════════════════
# 2. Class distribution (overall)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 75)
print("2. CLASS DISTRIBUTION (ALL SLIDES)")
print("=" * 75)

_slides_by_class = {}
_spectra_by_class = {}
for _s in _slides_in_h5:
    _c = _slide_class[_s]
    _slides_by_class.setdefault(_c, []).append(_s)
    _spectra_by_class[_c] = _spectra_by_class.get(_c, 0) + _spectra_count[_s]

_n_slides = len(_slides_in_h5)
_n_spectra = sum(_spectra_count.values())

print(f"\n  {'Class':<10} {'Slides':>8} {'% Slides':>10} {'Spectra':>12} {'% Spectra':>12}")
print(f"  {'-'*56}")
for _c in sorted(_slides_by_class):
    _ns = len(_slides_by_class[_c])
    _nsp = _spectra_by_class[_c]
    print(f"  {CLASS_NAMES[_c]:<10} {_ns:>8} {100*_ns/_n_slides:>9.1f}%"
          f" {_nsp:>12,} {100*_nsp/_n_spectra:>11.1f}%")
print(f"  {'TOTAL':<10} {_n_slides:>8} {'100.0%':>10} {_n_spectra:>12,} {'100.0%':>12}")

# ══════════════════════════════════════════════════════════════════════════
# 3. Per-fold statistics
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 75)
print("3. PER-FOLD CLASS DISTRIBUTION  (data/splits/manual_splits.json)")
print("=" * 75)

with open(_SPLITS_PATH) as _fp:
    _splits = json.load(_fp)

for _fold, _fold_data in sorted(_splits.items()):
    print(f"\n{'─'*75}")
    print(f"  {_fold.upper()}")
    print(f"{'─'*75}")
    for _split in ('train', 'test'):
        _names = _fold_data[_split]
        # split names already match sample_ids (e.g. 'HF-448_V5B_1.h5_3')
        _present = [n for n in _names if n in _slide_class]

        _sl_cls  = {}
        _sp_cls  = {}
        for _n in _present:
            _c = _slide_class[_n]
            _sl_cls[_c]  = _sl_cls.get(_c, 0)  + 1
            _sp_cls[_c]  = _sp_cls.get(_c, 0)  + _spectra_count.get(_n, 0)

        _tot_sl = len(_present)
        _tot_sp = sum(_sp_cls.values())
        print(f"\n  {_split.capitalize()} — {_tot_sl} slides, {_tot_sp:,} spectra")
        print(f"  {'Class':<10} {'Slides':>8} {'% Slides':>10} {'Spectra':>12} {'% Spectra':>12}")
        print(f"  {'-'*56}")
        for _c in sorted(set(_sl_cls) | set(_sp_cls)):
            _ns  = _sl_cls.get(_c, 0)
            _nsp = _sp_cls.get(_c, 0)
            _ps  = 100*_ns/_tot_sl  if _tot_sl  > 0 else 0
            _pp  = 100*_nsp/_tot_sp if _tot_sp > 0 else 0
            print(f"  {CLASS_NAMES.get(_c, str(_c)):<10} {_ns:>8} {_ps:>9.1f}%"
                  f" {_nsp:>12,} {_pp:>11.1f}%")
        _missing = [n for n in _names if n not in _slide_class]
        if _missing:
            print(f"  [not in HDF5: {', '.join(_missing)}]")